# Background Estimation — Solutions

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.stats     as stats  # statistics and Many PDFs 
#import scipy.optimize  as optimize # Minimice funtions

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

In [ ]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.efit    as efit     # Fit Utilites - Includes Extend Likelihood Fit with composite PDFs
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
import ana.pltfanal as pltfn    # plotting for fanal

import collpars    as collpars # collaboration specific parameters
pltext.style()

In [ ]:
coll          = collpars.collaboration
sel_erange    = collpars.sel_erange
sel_eroi      = collpars.sel_eroi
eff_Bi_E      = collpars.eff_Bi_E
eff_Tl_E      = collpars.eff_Tl_E
eff_Bi_RoI    = collpars.eff_Bi_RoI
eff_Tl_RoI    = collpars.eff_Tl_RoI

print('Collaboration         : {:s}'.format(coll))
print('Energy range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_erange))
print('RoI    range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_eroi))
print('Bi eff in E range     : {:6.4f} %'.format(100.*eff_Bi_E))
print('Tl eff in E range     : {:6.4f} %'.format(100.*eff_Tl_E))
print('Bi eff in RoI         : {:6.4f} %'.format(100.*eff_Bi_RoI))
print('Tl eff in RoI         : {:6.4f} %'.format(100.*eff_Tl_RoI))

In [ ]:
#dirpath = '/Users/hernando/docencia/master/Fisica_Particulas/USC-Fanal/data/'
filename = '/data/fanal_' + coll + '.h5'
print('Data path and filename : ', rootpath + filename)

mcbi       = pd.read_hdf(rootpath + filename, key = 'mc/bi214').fillna(0.)   # MC Bi
mctl       = pd.read_hdf(rootpath + filename, key = 'mc/tl208').fillna(0.)   # MC Tl
data_blind = pd.read_hdf(rootpath + filename, key = 'data/blind').fillna(0.) # blind data

mc_samples         = [mcbi, mctl]
sample_names       = ['Bi', 'Tl']
sample_names_latex = [r'$^{214}$Bi', r'$^{208}$Tl']

In [ ]:
# apply the blind selection to the mc samples to obtaine blind mc samples

def efficiency(mask):
    return float(sum(mask)/len(mask))

mask_bi = fn.selection_blind(mcbi)
mask_tl = fn.selection_blind(mctl)

eff_Bi_blind = efficiency(mask_bi)
eff_Tl_blind = efficiency(mask_tl)

mc_blind_samples =  (mcbi[mask_bi], mctl[mask_tl])

print('Bi blind eff : {:6.4f} %'.format(100.*eff_Bi_blind))
print('Tl blind edd : {:6.4f} %'.format(100.*eff_Tl_blind))

In [ ]:
nguess = (1000., 1000.)

varname   = 'E'
refnames  = (varname,) 
refranges = (sel_erange,)

fit        = fn.prepare_fit_ell(mc_blind_samples, nguess, refnames, refranges)

result, enes, ell, _ = fit(data_blind)
nevts_blind          = result.x

pltfn.plot_fit_ell(enes, nevts_blind, ell, parnames = sample_names_latex);

In [ ]:
n_Bi_blind = nevts_blind[0]
n_Tl_blind = nevts_blind[1]

print(' number of blind events Bi : {:6.3f}'.format(n_Bi_blind))
print(' number of blind events Tl : {:6.3f}'.format(n_Tl_blind))

## Compute total, energy-range and RoI background event counts

In [ ]:
n_Bi_total = n_Bi_blind/eff_Bi_blind
n_Tl_total = n_Tl_blind/eff_Tl_blind

n_Bi_E = n_Bi_total * eff_Bi_E
n_Tl_E = n_Tl_total * eff_Tl_E

n_Bi_RoI = n_Bi_total * eff_Bi_RoI
n_Tl_RoI = n_Tl_total * eff_Tl_RoI

In [ ]:
print(' number of events total Bi      : {:6.3f}'.format(n_Bi_total))
print(' number of events total Tl      : {:6.3f}'.format(n_Tl_total))

print(' number of events in E range Bi : {:6.3f}'.format(n_Bi_E))
print(' number of events in E range Tl : {:6.3f}'.format(n_Tl_E))

print(' number of events in RoI Bi     : {:6.3f}'.format(n_Bi_RoI))
print(' number of events in RoI Tl     : {:6.3f}'.format(n_Tl_RoI))

### Notation-to-code reference

| Math | Python variable | Description |
|------|-----------------|-------------|
| — | `exposure` | Detector exposure (kg $\cdot$ y) |
| — | `acc_bb` | Signal acceptance |
| $\epsilon^\mathrm{Bi}_b$ | `eff_Bi_blind` | $^{214}$Bi blind selection efficiency |
| $\epsilon^\mathrm{Tl}_b$ | `eff_Tl_blind` | $^{208}$Tl blind selection efficiency |
| $n^\mathrm{Bi}_b$ | `n_Bi_blind` | $^{214}$Bi events in blind sample |
| $n^\mathrm{Tl}_b$ | `n_Tl_blind` | $^{208}$Tl events in blind sample |
| $n^\mathrm{Bi}_\text{total}$ | `n_Bi_total` | Total $^{214}$Bi events |
| $n^\mathrm{Tl}_\text{total}$ | `n_Tl_total` | Total $^{208}$Tl events |
| $n^\mathrm{Bi}_E$ | `n_Bi_E` | $^{214}$Bi events in energy range |
| $n^\mathrm{Tl}_E$ | `n_Tl_E` | $^{208}$Tl events in energy range |
| $n^\mathrm{Bi}_{RoI}$ | `n_Bi_RoI` | $^{214}$Bi events in RoI |
| $n^\mathrm{Tl}_{RoI}$ | `n_Tl_RoI` | $^{208}$Tl events in RoI |

In [ ]:
import ana.fanal_display as fdisp

exposures  = {'new_alpha': 500, 'new_beta': 1000, 'new_gamma': 1000,
              'new_delta': 3000, 'new_epsilon': 3000}
acc_bb     = 0.79
exposure   = exposures[coll]

fdisp.display_collpars([
    ('—',                          'exposure',     exposure,     '.2f'),
    ('—',                          'acc_bb',       acc_bb,       '.3f'),
    (r'\epsilon^\mathrm{Bi}_b',    'eff_Bi_blind', eff_Bi_blind, '.3f'),
    (r'\epsilon^\mathrm{Tl}_b',    'eff_Tl_blind', eff_Tl_blind, '.3f'),
    (r'n^\mathrm{Bi}_b',           'n_Bi_blind',   n_Bi_blind,   '.3f'),
    (r'n^\mathrm{Tl}_b',           'n_Tl_blind',   n_Tl_blind,   '.3f'),
    (r'n^\mathrm{Bi}_\text{total}','n_Bi_total',   n_Bi_total,   '.3f'),
    (r'n^\mathrm{Tl}_\text{total}','n_Tl_total',   n_Tl_total,   '.3f'),
    (r'n^\mathrm{Bi}_E',           'n_Bi_E',       n_Bi_E,       '.3f'),
    (r'n^\mathrm{Tl}_E',           'n_Tl_E',       n_Tl_E,       '.3f'),
    (r'n^\mathrm{Bi}_{RoI}',       'n_Bi_RoI',     n_Bi_RoI,     '.3f'),
    (r'n^\mathrm{Tl}_{RoI}',       'n_Tl_RoI',     n_Tl_RoI,     '.3f'),
])

### Write out

In [ ]:
exposures  = {'new_alpha': 500, 'new_beta': 1000, 'new_gamma': 1000,
              'new_delta': 3000, 'new_epsilon': 3000}
acc_bb     = 0.79

exposure   = exposures[coll]

write = True
if (write):
    of = open('collpars.py', 'a')
    of.write('exposure        = {:6.2f}'.format(exposure)+' # kg y \n')
    of.write('acc_bb          = {:6.3f}'.format(acc_bb)+'\n')

    of.write('eff_Bi_blind    = {:6.3f}'.format(eff_Bi_blind)+'\n')
    of.write('eff_Tl_blind    = {:6.3f}'.format(eff_Tl_blind)+'\n')

    of.write('n_Bi_total      = {:6.3f}'.format(n_Bi_total)+'\n')
    of.write('n_Bi_blind      = {:6.3f}'.format(n_Bi_blind)+'\n')
    of.write('n_Bi_E          = {:6.3f}'.format(n_Bi_E)+'\n')
    of.write('n_Bi_RoI        = {:6.3f}'.format(n_Bi_RoI)+'\n')

    of.write('n_Tl_total      = {:6.3f}'.format(n_Tl_total)+'\n')
    of.write('n_Tl_blind      = {:6.3f}'.format(n_Tl_blind)+'\n')
    of.write('n_Tl_E          = {:6.3f}'.format(n_Tl_E)+'\n')
    of.write('n_Tl_RoI        = {:6.3f}'.format(n_Tl_RoI)+'\n')
    of.close()